## Employee Attrition Prediction

\importing necessary libraries

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

from sklearn.model_selection import train_test_split

\loading the merged dataset

In [13]:
df = pd.read_csv(r"C:\NG\Employee Attrition Prediction\data\merged_data\employee_merged.csv")
df.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeID,Gender,JobLevel,JobRole,MaritalStatus,MonthlyIncome,NumCompaniesWorked,Over18,PercentSalaryHike,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,YearsAtCompany,YearsSinceLastPromotion,YearsWithCurrManager,EnvironmentSatisfaction,JobSatisfaction,WorkLifeBalance,JobInvolvement,PerformanceRating,avg_work_hours
0,51,No,Travel_Rarely,Sales,6,2,Life Sciences,1,1,Female,1,Healthcare Representative,Married,131160,1.0,Y,11,8,0,1.0,6,1,0,0,3.0,4.0,2.0,3,3,7.37
1,31,Yes,Travel_Frequently,Research & Development,10,1,Life Sciences,1,2,Female,1,Research Scientist,Single,41890,0.0,Y,23,8,1,6.0,3,5,1,4,3.0,2.0,4.0,2,4,7.72
2,32,No,Travel_Frequently,Research & Development,17,4,Other,1,3,Male,4,Sales Executive,Married,193280,1.0,Y,15,8,3,5.0,2,5,0,3,2.0,2.0,1.0,3,3,7.01
3,38,No,Non-Travel,Research & Development,2,5,Life Sciences,1,4,Male,3,Human Resources,Married,83210,3.0,Y,11,8,3,13.0,5,8,7,5,4.0,4.0,3.0,2,3,7.19
4,32,No,Travel_Rarely,Research & Development,10,1,Medical,1,5,Male,1,Sales Executive,Single,23420,4.0,Y,12,8,2,9.0,2,6,0,4,4.0,1.0,3.0,3,3,8.01


In [14]:
df.shape

(4410, 30)

\train test split

In [15]:
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

X = df.drop(columns=['Attrition'])
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Split done ")
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("\nTrain target %:")
print((y_train.value_counts(normalize=True)*100).round(2).to_dict())
print("Test target %:")
print((y_test.value_counts(normalize=True)*100).round(2).to_dict())

Split done 
X_train: (3528, 29) | X_test: (882, 29)

Train target %:
{0: 83.87, 1: 16.13}
Test target %:
{0: 83.9, 1: 16.1}


\data preprocessing

In [16]:
drop_cols = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeID']

X_train = X_train.drop(columns=drop_cols)
X_test  = X_test.drop(columns=drop_cols)

print("Dropped useless columns ")
print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)
print("\nRemaining columns:", list(X_train.columns))

Dropped useless columns 
X_train shape: (3528, 25)
X_test  shape: (882, 25)

Remaining columns: ['Age', 'BusinessTravel', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'Gender', 'JobLevel', 'JobRole', 'MaritalStatus', 'MonthlyIncome', 'NumCompaniesWorked', 'PercentSalaryHike', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'YearsAtCompany', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'EnvironmentSatisfaction', 'JobSatisfaction', 'WorkLifeBalance', 'JobInvolvement', 'PerformanceRating', 'avg_work_hours']


\missing value imputation

In [17]:
mode_cols   = ['EnvironmentSatisfaction', 'JobSatisfaction', 'WorkLifeBalance']  # survey scores
median_cols = ['NumCompaniesWorked', 'TotalWorkingYears']                        # numeric counts

fill_values = {}
for col in mode_cols:
    fill_values[col] = X_train[col].mode()[0]     
for col in median_cols:
    fill_values[col] = X_train[col].median()      

print("Fill values learned from TRAIN:")
print(fill_values)

X_train = X_train.fillna(fill_values)
X_test  = X_test.fillna(fill_values)

print("\nMissing in X_train:", X_train.isnull().sum().sum())
print("Missing in X_test :", X_test.isnull().sum().sum())

Fill values learned from TRAIN:
{'EnvironmentSatisfaction': np.float64(3.0), 'JobSatisfaction': np.float64(4.0), 'WorkLifeBalance': np.float64(3.0), 'NumCompaniesWorked': np.float64(2.0), 'TotalWorkingYears': np.float64(10.0)}

Missing in X_train: 0
Missing in X_test : 0


In [18]:
print("Data types after cleaning:")
print(X_train.dtypes.value_counts())
print("\nCategorical (object) columns:")
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
print(cat_cols)

Data types after cleaning:
int64      13
str         6
float64     6
Name: count, dtype: int64

Categorical (object) columns:
['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus']


\outlier detection

In [19]:
numeric_cols = X_train.select_dtypes(include='number').columns.tolist()

outlier_summary = {}
for col in numeric_cols:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((X_train[col] < lower) | (X_train[col] > upper)).sum()
    pct = round(n_out / len(X_train) * 100, 2)
    outlier_summary[col] = {'lower': round(lower,2), 'upper': round(upper,2),
                            'n_outliers': n_out, 'pct': pct}


In [20]:
outlier_df = pd.DataFrame(outlier_summary).T.sort_values('n_outliers', ascending=False)
print("Outlier summary (IQR method, TRAIN):\n")
outlier_df

Outlier summary (IQR method, TRAIN):



,lower,upper,n_outliers,pct
TrainingTimesLastYear,0.50,4.50,556.0,15.76
PerformanceRating,3.00,3.00,555.0,15.73
YearsSinceLastPromotion,-3.00,5.00,511.0,14.48
MonthlyIncome,-53152.50,165547.50,276.0,7.82
YearsAtCompany,-6.00,18.00,241.0,6.83
StockOptionLevel,-1.50,2.50,201.0,5.70
TotalWorkingYears,-7.50,28.50,153.0,4.34
NumCompaniesWorked,-3.50,8.50,130.0,3.68
YearsWithCurrManager,-5.50,14.50,33.0,0.94
avg_work_hours,4.11,10.96,23.0,0.65


### Outlier Treatment Decision

Findings:
- Outliers mainly in: YearsSinceLastPromotion, YearsAtCompany, 
  TotalWorkingYears (all < 3%)
- These are GENUINE values — senior/long-tenured employees really exist
- Such employees are important attrition signals

Decision: KEEP outliers (no removal/capping).
Reasons:
1. They are real, valid HR values 
2. Removing = losing real attrition patterns
3. Our main models (RF, Gradient Boosting, XGBoost) are 
   tree-based → naturally robust to outliers
4. Only Logistic Regression is sensitive → scaling
   will reduce their impact
